In [ ]:
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer

from pymilvus import MilvusClient

from tqdm import tqdm
import json
import os
from pathlib import Path

C:\Users\amir1\AppData\Local\Temp\ipykernel_18204\3373764571.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
c:\AMORE\Development\Python Projects\yet-another-RAG-assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [21]:
def document_load(file_path):
    file = Path(file_path)

    if file.suffix.lower() == ".pdf":
        loader = PyPDFLoader(str(file))
        docs = loader.load()

        print("Документ формата pdf был загружен")
        return docs

    elif file.suffix.lower() == ".docx":
        loader = Docx2txtLoader(str(file))
        docs = loader.load()

        print("Документ формата docx был загружен")
        return docs

    raise ValueError(f"Формат {file.suffix} не поддерживается")

In [22]:
def chunk_docs(docs, chunk_size, chunk_overlap):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunks = splitter.split_documents(docs)

    print(f"Чанкинг завершён. Получено {len(chunks)} чанков")

    return chunks

In [23]:
def create_collection(
    milvus_client,
    collection_name,
    embedding_dim,
    metric_type="IP"
):
    if milvus_client.has_collection(collection_name):
        milvus_client.drop_collection(collection_name)

    milvus_client.create_collection(
        collection_name=collection_name,
        dimension=embedding_dim,
        metric_type=metric_type,
    )

    print(f"Коллекция {collection_name} создана")

In [24]:
def emb_text(text, embedding_model):
    return embedding_model.encode([text], normalize_embeddings=True).tolist()[0]

In [25]:
def vectordb_insert(
    milvus_client,
    chunks,
    collection_name,
    embedding_model
):
    text_lines = [chunk.page_content for chunk in chunks]
    
    data = []

    for i, line in enumerate(tqdm(text_lines, desc="Создание эмбеддингов...")):
        data.append({"id": i, "vector": emb_text(line, embedding_model), "text": line})

    milvus_client.insert(
        collection_name=collection_name,
        data=data
    )

    print(
        f"Сохранено {len(data)} документов "
        f"в коллекцию {collection_name}"
    )

In [26]:
def load_ingest_pipeline(
    file_path,
    chunk_size,
    chunk_overlap,
    milvus_client,
    collection_name,
    metric_type,
    embedding_model_name
):
    docs = document_load(file_path)

    chunks = chunk_docs(
        docs,
        chunk_size,
        chunk_overlap
    )

    embedding_model = SentenceTransformer(
        embedding_model_name,
        cache_folder="models_cache"
    )

    print("Модель эмбеддингов загружена")

    embedding_dim = embedding_model.get_embedding_dimension()

    create_collection(
        milvus_client,
        collection_name,
        embedding_dim,
        metric_type
    )

    vectordb_insert(
        milvus_client,
        chunks,
        collection_name,
        embedding_model
    )

    return "Пайплайн прошёл успешно!"

In [27]:
file_path = r"C:\AMORE\Development\Python Projects\yet-another-RAG-assistant\data\documents\чаво_2.pdf"

milvus_client = MilvusClient(
    uri="http://localhost:19530"
)

In [28]:
load_ingest_pipeline(
    file_path=file_path,
    chunk_size=900,
    chunk_overlap=100,
    milvus_client=milvus_client,
    collection_name="RAG_col",
    metric_type="IP",
    embedding_model_name="intfloat/multilingual-e5-small"
)

Документ формата pdf был загружен
Чанкинг завершён. Получено 13 чанков


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4260.98it/s]


Модель эмбеддингов загружена
Коллекция RAG_col создана


Создание эмбеддингов...: 100%|██████████| 13/13 [00:01<00:00, 11.10it/s]


Сохранено 13 документов в коллекцию RAG_col


'Пайплайн прошёл успешно!'